# Example 1: Source Discovery

Find the right AIS data source for your workflow.

Neptune supports multiple AIS data sources with different coverage,
delivery modes, and access requirements. This notebook shows how to
inspect what is available before downloading anything.

## Prerequisites

```bash
pip install neptune-ais[notebooks]
```

This example is fully self-contained — no network access, API keys, or external data required.

## Imports

In [6]:
from neptune_ais.adapters.registry import (
    load_all_adapters,
    catalog,
    info,
    compare,
    find_sources,
)

## Load Adapters

Before querying the source catalog, load all built-in adapters and any installed plugins.
This discovers every adapter registered via Python entry points.

In [7]:
load_all_adapters()

['aishub', 'aisstream', 'dma', 'finland', 'gfw', 'noaa']

## List All Registered Sources

The `catalog()` function returns a `SourceCapabilities` object for every registered adapter,
giving you a quick overview of available data providers.

In [8]:
print("=== All registered sources ===")
for caps in catalog():
    print(f"  {caps.source_id:<12} {caps.provider:<30} {caps.coverage}")

=== All registered sources ===
  aishub       AISHub                         Global terrestrial (community receivers)
  dma          Danish Maritime Authority      Danish waters / Western Baltic Sea
  finland      Fintraffic Digitraffic         Finnish waters and Baltic Sea
  gfw          Global Fishing Watch           Global
  noaa         NOAA Marine Cadastre           U.S. coastal waters
  aisstream    aisstream.io                   Global (terrestrial AIS receivers)


## Inspect a Specific Source

Use `info()` to get detailed metadata for a single source, including
coverage area, history start date, authentication requirements, and license terms.

In [9]:
noaa = info("noaa")
print(f"Provider:  {noaa.provider}")
print(f"Coverage:  {noaa.coverage}")
print(f"History:   {noaa.history_start}")
print(f"Backfill:  {noaa.supports_backfill}")
print(f"Auth:      {noaa.auth_scheme or 'none'}")
print(f"License:   {noaa.license_requirements}")

Provider:  NOAA Marine Cadastre
Coverage:  U.S. coastal waters
History:   2009-01-01
Backfill:  True
Auth:      none
License:   Public domain (U.S. Government work)


## Compare Sources Side-by-Side

`compare()` produces a structured comparison of two or more sources,
making it easy to choose between providers for a given use case.

In [10]:
for summary in compare("noaa", "gfw"):
    print(f"  {summary['source']:<8} backfill={summary['backfill']}, "
          f"auth={summary['auth']}, coverage={summary['coverage']}")

  noaa     backfill=yes, auth=none, coverage=U.S. coastal waters
  gfw      backfill=yes, auth=api_key, coverage=Global


## Filter Sources by Capability

`find_sources()` filters the catalog by properties like authentication
requirements, streaming support, and backfill availability.

In [11]:
print("=== Open-data sources (no authentication required) ===")
for caps in find_sources(auth=False):
    print(f"  {caps.source_id}: {caps.provider}")

=== Open-data sources (no authentication required) ===
  dma: Danish Maritime Authority
  finland: Fintraffic Digitraffic
  noaa: NOAA Marine Cadastre


In [12]:
print("=== Sources supporting streaming ===")
for caps in find_sources(streaming=True):
    print(f"  {caps.source_id}: {caps.provider}")

=== Sources supporting streaming ===
  finland: Fintraffic Digitraffic
  aisstream: aisstream.io


In [13]:
print("=== Sources with backfill + no auth ===")
for caps in find_sources(backfill=True, auth=False):
    print(f"  {caps.source_id}: {caps.provider}")

=== Sources with backfill + no auth ===
  dma: Danish Maritime Authority
  finland: Fintraffic Digitraffic
  noaa: NOAA Marine Cadastre


## Next Steps

Now that you know which sources are available, proceed to:

- **[02 — Archival Ingest](02_archival_ingest.ipynb)**: Download data and query it with Polars and SQL
- **[03 — Multi-Source Fusion](03_multi_source_fusion.ipynb)**: Combine data from multiple providers